# 基礎編13
このノートブックでは、トランザクションログを検索する例を示します。

## 準備

実行の準備を行います。

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { loadWallet } = await import('../lib/load-wallet.mjs');
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

あらかじめ用意してあるユーザ／ウォレットを読み込みます。

In [2]:
var users = [];
for(var i=0; i<10; i++){
    var wfstr = await loadWallet(`wallet-user${i}.json`);
    var wallet = await api.unlockWalletFile(await api.parseWalletFile(wfstr), '_paswrd_');
    var id = (await rpc.call(wallet, 'c1query', { type: 'search', key: 'me' })).value[0].id;
    users[i] = { id, wallet };
    console.log(`user${i}:`, id, wallet.address);
}

user0: u73621973 eQiA3bsfZunQ8KsRpkTgMGndwu46rq
user1: u28169743 epGyEm4BXFxFvLmXkDdDvcqspESPFu
user2: u53371386 eaZMspgRpi68EdReTZEiUkdJuKsdct
user3: u58185320 e5fVm5R4eKYXa7U57pshS4UQwuvGsp
user4: u10676071 eEYRycGm4znXxnf7bTngX72JmYk57r
user5: u00571239 eWW6LCfRSpoVuudzZHa4ZwCjkkizov
user6: u78093328 eYKwuqvDkdBAktANt6oebrdYmStexw
user7: u18531789 eZbEh77Aq8RpmpZ6bgcTEYVGiYYNsv
user8: u48750131 eX5jgq78UcBfUZg9SmnXFim6zLnf9r
user9: u13917166 eAvw2Hf4TSreW2Sfj3PFUmXADUSZJg


## スマートコントラクトのデプロイ

引数a,bを受け付け、relateToを実行する簡単なスマートコントラクトを作成します。


In [3]:
var cid = await deploySmartContract({ a: 'string', b: 'number' }, function basic13(a, b){
    relateTo('kwd'+b);
});
console.log(cid);

c028539995


作成したスマートコントラクトを上記ユーザが実行できるように許可します。

In [4]:
await rpc.call(adminWallet, 'c1update', { id: cid, prop: 'add accessible_to', value: users.map(e => e.id) });

{
  txno: 701815,
  txid: 'xEaQLaWUuerJR2eazNo9ZLkJsfTqVDTanEKzt8Y2sCzCFB',
  status: 'ok',
  value: null
}

作成したスマートコントラクトのトランザクションログをadminWalletで参照できるように許可します。

In [5]:
var id = (await rpc.call(adminWallet, 'c1query', { type: 'search', key: 'me' })).value[0].id;
await rpc.call(adminWallet, 'c1update', { id: cid, prop: 'add disclosed_to', value: [ id ] });

{
  txno: 701816,
  txid: 'xHRq28RBZgwPuffmyMHanjwhN9G9NeaEvLRBEumsRpMzw',
  status: 'ok',
  value: null
}

## スマートコントラクトの実行

ウォレットや引数を変えながらトランザクションを合計100個実行します。

In [6]:
for(var j=0; j<10; j++){
    for (var i=0; i<10; i++){
        var resp = await rpc.call(users[i].wallet, cid, { a: 'str'+j , b: i+j });
    }
}
console.log(resp);

{
  txno: 701916,
  txid: 'xBFAGoQ6yCyiA9CFRzpXb73WXe2Ebjjz3MLbHRyv8sPew',
  status: 'ok',
  value: null
}


## トランザクションログの一覧の取得

c1query-transactionsを使って、トランザクションの一覧を取得します。 
一覧は記録が新しい順に並びます。  
ここでは、出力が長くなるのを避けるため、最大出力数を5に制限しています。  

In [7]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'transactions', limit: 5 });
console.log(resp);

{
  txid: 'xaCfgPXX9AMJVTfQPG73xiSgcYYpFp5oS7frmWWhQcrhQB',
  status: 'ok',
  value: {
    list: [
      {
        txno: 701916,
        caller: [ 'u13917166', '_test_user9@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420904434
      },
      {
        txno: 701915,
        caller: [ 'u48750131', '_test_user8@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420904083
      },
      {
        txno: 701914,
        caller: [ 'u18531789', '_test_user7@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420903732
      },
      {
        txno: 701913,
        caller: [ 'u78093328', '_test_user6@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420903383
      },
      {
        txno: 701912,
        caller: [ 'u00571239', '_test_user5@c.T

続きがある場合には、value.moreに続きのtxnoが格納されています。  
続きを取得したい場合には、before_txno条件にvalue.moreの値を指定して再度問い合わせます。

In [8]:
var more = resp.value.more;
var resp = await rpc.call(adminWallet, 'c1query', { type: 'transactions', limit: 5, before_txno: more });
console.log(resp);

{
  txid: 'xy6pzZ8XGikSzJbhQHFQSotasVGx6gSsFvMwrYiavnVLs',
  status: 'ok',
  value: {
    list: [
      {
        txno: 701911,
        caller: [ 'u10676071', '_test_user4@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420902677
      },
      {
        txno: 701910,
        caller: [ 'u58185320', '_test_user3@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420902324
      },
      {
        txno: 701909,
        caller: [ 'u53371386', '_test_user2@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420901971
      },
      {
        txno: 701908,
        caller: [ 'u28169743', '_test_user1@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420901621
      },
      {
        txno: 701907,
        caller: [ 'u73621973', '_test_user0@c.TD

トランザクションログの引数もあわせて取得したい場合には、detailsに'argstr'を指定します。  
なお、argstrは引数オブジェクトのJSON文字列です。  

In [9]:
var more = resp.value.more;
var resp = await rpc.call(adminWallet, 'c1query', { type: 'transactions', limit: 5, details: ['argstr'], before_txno: more });
console.log(resp);

{
  txid: 'xgN49LdfXAMXNfBUg5vnBW5yvGaqtcdFnvpEYwcQNyrWo',
  status: 'ok',
  value: {
    list: [
      {
        txno: 701906,
        caller: [ 'u13917166', '_test_user9@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420900916,
        argstr: '{"a":"str8","b":17}'
      },
      {
        txno: 701905,
        caller: [ 'u48750131', '_test_user8@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420900566,
        argstr: '{"a":"str8","b":16}'
      },
      {
        txno: 701904,
        caller: [ 'u18531789', '_test_user7@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420900215,
        argstr: '{"a":"str8","b":15}'
      },
      {
        txno: 701903,
        caller: [ 'u78093328', '_test_user6@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok'

## コントラクトによる絞り込み

呼び出されたコントラクトによって絞り込むことができます。  
ここでは例として、c1updateコントラクトが呼び出されたトランザクションに絞り込みます。

In [10]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'transactions', limit: 5, details: ['argstr'], callee: 'c1update' });
console.log(resp);

{
  txid: 'xPEpT8uxpFombhrL6dT6uiY9phf9NRJGYcwAaBsfkbGY3',
  status: 'ok',
  value: {
    list: [
      {
        txno: 701816,
        caller: [ 'u58281888', 'jupyter@c.TDSL.H011' ],
        callee: [ 'c1update', 'c1update@default' ],
        status: 'ok',
        time: 1753420868764,
        argstr: '{"id":"c028539995","prop":"add disclosed_to","value":["u58281888"]}'
      },
      {
        txno: 701815,
        caller: [ 'u58281888', 'jupyter@c.TDSL.H011' ],
        callee: [ 'c1update', 'c1update@default' ],
        status: 'ok',
        time: 1753420868077,
        argstr: '{"id":"c028539995","prop":"add accessible_to","value":["u73621973","u28169743","u53371386","u58185320","u10676071","u00571239","u78093328","u18531789","u48750131","u13917166"]}'
      },
      {
        txno: 701814,
        caller: [ 'u58281888', 'jupyter@c.TDSL.H011' ],
        callee: [ 'c1update', 'c1update@default' ],
        status: 'ok',
        time: 1753420867496,
        argstr: '{"id":"c028539995",

## 呼び出しユーザによる絞り込み

呼び出したユーザによって絞り込むことができます。  
ここでは例として、user5が呼び出したトランザクションに絞り込みます。

In [11]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'transactions', limit: 5, caller: users[5].id });
console.log(resp);

{
  txid: 'xfXG5Td5wxFFq556Zg2FobGMEcHqPeY6zwpTDbcWkBepn',
  status: 'ok',
  value: {
    list: [
      {
        txno: 701912,
        caller: [ 'u00571239', '_test_user5@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420903030
      },
      {
        txno: 701902,
        caller: [ 'u00571239', '_test_user5@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420899504
      },
      {
        txno: 701892,
        caller: [ 'u00571239', '_test_user5@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420895983
      },
      {
        txno: 701882,
        caller: [ 'u00571239', '_test_user5@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420892459
      },
      {
        txno: 701872,
        caller: [ 'u00571239', '_test_user5@c.TD

## 引数による絞り込み

引数の値によって絞り込むことができます。  
ここでは例として、上記作成したコントラクトの引数bの値が15であるトランザクションに絞り込みます。  

In [12]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'transactions', limit: 5, details: ['argstr'], callee: cid, arg: [ 'b', '==', 15 ] });
console.log(resp);

{
  txid: 'x9pymNZ5JNq4XKjvaZv5x7fpP8eJ5r3DuPJPWYW74o4q5',
  status: 'ok',
  value: {
    list: [
      {
        txno: 701913,
        caller: [ 'u78093328', '_test_user6@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420903383,
        argstr: '{"a":"str9","b":15}'
      },
      {
        txno: 701904,
        caller: [ 'u18531789', '_test_user7@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420900215,
        argstr: '{"a":"str8","b":15}'
      },
      {
        txno: 701895,
        caller: [ 'u48750131', '_test_user8@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420897039,
        argstr: '{"a":"str7","b":15}'
      },
      {
        txno: 701886,
        caller: [ 'u13917166', '_test_user9@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok'

## 検索ワードによる絞り込み

検索ワードによって絞り込むことができます。  
検索ワードは、コントラクトの実行中にrelateTo()で指定することができます。  
ここでは例として、検索ワードとしてkwd12が設定されているトランザクションに絞り込みます。

In [13]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'transactions', limit: 5, details: ['argstr'], related_to: 'kwd12' });
console.log(resp);

{
  txid: 'xKFZjFsTFk5ZwjboZPQCA34X6akQNHdzymtfRwQByRxor',
  status: 'ok',
  value: {
    list: [
      {
        txno: 701910,
        caller: [ 'u58185320', '_test_user3@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420902324,
        argstr: '{"a":"str9","b":12}'
      },
      {
        txno: 701901,
        caller: [ 'u10676071', '_test_user4@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420899153,
        argstr: '{"a":"str8","b":12}'
      },
      {
        txno: 701892,
        caller: [ 'u00571239', '_test_user5@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420895983,
        argstr: '{"a":"str7","b":12}'
      },
      {
        txno: 701883,
        caller: [ 'u78093328', '_test_user6@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok'

## 複数の絞り込み条件

複数の絞り込み条件が指定された場合には、AND条件となります。

In [14]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'transactions', limit: 5, related_to: 'kwd12', caller: users[9].id, callee: cid });
console.log(resp);

{
  txid: 'xPbST4REfzj3S7y63FeqTZwhyEui3hDuAWa2w95Pkib9p',
  status: 'ok',
  value: {
    list: [
      {
        txno: 701856,
        caller: [ 'u13917166', '_test_user9@c.TDSL.H011' ],
        callee: [ 'c028539995', 'basic13@c.TDSL.H011' ],
        status: 'ok',
        time: 1753420883272
      }
    ]
  }
}
